In [1]:
# ============================================================
# One-shot inference pass: dump validation-set MEL/BKL/NV
# probabilities from the ALREADY-TRAINED fixed-weight
# SSL + rebalance checkpoints. No retraining. Run this once
# per seed (or loop it), then upload the three resulting CSVs.
#
# Fill in CKPT_PATTERN to match wherever your fixed-weight
# checkpoints actually live -- likely inside the same Kaggle
# output/dataset that produced test_probs_seed*.csv.
# ============================================================
import os, glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- EDIT THESE THREE LINES ----
BACKUP_DIR   = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
LABELED_DIR  = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
CKPT_PATTERN = '/kaggle/input/datasets/ittisamurtunib/ssl-3seed-pth/ssl_rebalance_seed{seed}_finetuned.pth'
# ---------------------------------

OUTPUT_DIR = '/kaggle/working'
SEEDS = [42, 123, 2024]

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class EvalDataset(Dataset):
    def __init__(self, df, transform):
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['image'])
        if not img_name.endswith('.jpg'):
            img_name += '.jpg'
        img_path = os.path.join(LABELED_DIR, img_name)
        if not os.path.exists(img_path):
            m = glob.glob(f"{LABELED_DIR}/**/{img_name}", recursive=True)
            img_path = m[0] if m else img_path
        img = self.transform(Image.open(img_path).convert('RGB'))
        return img, row['label']

class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

val_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
CLASS_ORDER = sorted(val_df['label'].unique())  # e.g. ['BKL','MEL','NV'] -- must match training order

for seed in SEEDS:
    ckpt_path = CKPT_PATTERN.format(seed=seed)
    assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"

    ds = EvalDataset(val_df, test_transform)
    loader = DataLoader(ds, batch_size=16, shuffle=False)

    model = SSLClassifier(SimpleEncoder(), len(CLASS_ORDER)).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    rows = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            probs = torch.softmax(model(images), dim=1).cpu().numpy()
            for p, lbl in zip(probs, labels):
                row = {c: float(p[i]) for i, c in enumerate(CLASS_ORDER)}
                row['true_label'] = lbl
                rows.append(row)

    out_df = pd.DataFrame(rows)[CLASS_ORDER + ['true_label']]
    out_path = os.path.join(OUTPUT_DIR, f'val_probs_seed{seed}.csv')
    out_df.to_csv(out_path, index=False)
    print(f"Seed {seed}: wrote {len(out_df)} rows -> {out_path}")

Seed 42: wrote 104 rows -> /kaggle/working/val_probs_seed42.csv
Seed 123: wrote 104 rows -> /kaggle/working/val_probs_seed123.csv
Seed 2024: wrote 104 rows -> /kaggle/working/val_probs_seed2024.csv
